In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/global/cfs/cdirs/desi/spectro/redux/everest/exposures-everest.fits', ext='EXPOSURES'))
print(len(cat), len(np.unique(cat['EXPID'])))

3912 3912


In [4]:
subsamp_dict = {'80605': [[73702, 74779, 68291, 74780, 68292, 74782, 67972]],
 '80606': [[67970, 68813, 68285, 68289, 68630, 67968, 67971]],
 '80607': [[67767, 68028, 68847, 68664, 68845, 68846],
  [67765, 68662, 68663, 68027, 67766, 68666]],
 '80608': [[68842, 69252, 68023, 68024, 68661],
  [69253, 68491, 67769, 69249, 67770, 68328, 67771],
  [68660, 68327, 68317, 68025, 68841]],
 '80609': [[68490, 68339, 68337, 68334, 68065, 68336],
  [67781, 68063, 68064, 67783, 68340, 68489, 68338]],
 '80610': [[68332, 68478, 68041, 68330, 68488, 68683, 75114],
  [75113, 68477, 68040, 68333, 68331, 68042, 75116]],
 '80613': [[69226, 68658, 69230, 69228],
  [81833, 81834],
  [81835, 69227, 68657],
  [69229, 81832, 69225, 68659, 81831]]}

In [5]:
# exp_list = sorted([i for k in subsamp_dict.values() for j in k for i in j])
# print(len(exp_list), len(np.unique(exp_list)))

# mask = np.in1d(cat['EXPID'], exp_list)
# print(np.sum(mask))
# # cat[mask]

In [6]:
mask_bgs = cat['TILEID']==80613
mask_bgs &= cat['BGS_EFFTIME_BRIGHT']/cat['EXPTIME'] > 0.2

mask_dark = cat['TILEID']!=80613
mask_dark &= cat['EFFTIME_SPEC']/cat['EXPTIME'] > 0.3

mask = mask_bgs | mask_dark
cat = cat[mask]
print(len(cat))

2231


In [7]:
print('                 ELG_EFFTIME_DARK  BGS_EFFTIME_BRIGHT   EFFTIME_DARK_GFA   EFFTIME_BRIGHT_GFA')
print()

# for tileid in subsamp_dict.keys():
for tileid in ['80605', '80607', '80609', '80606', '80608', '80610', '80613']:
    mask = cat['TILEID']==int(tileid)
    print(cat['FAPRGRM'][mask][0].upper(), tileid)
    for subset_index, exp_list in enumerate(subsamp_dict[tileid]):
        # print(exp_list)
        mask = np.in1d(cat['EXPID'], exp_list)
        if np.sum(mask)>1:
            print_str = 'exposures'
        else:
            print_str = 'exposure '
        print('subset {}: {} {} {:11.0f} {:19.0f} {:18.0f} {:20.0f}'.format(subset_index+1, np.sum(mask), print_str, np.sum(cat['ELG_EFFTIME_DARK'][mask]), np.sum(cat['BGS_EFFTIME_BRIGHT'][mask]), np.sum(cat['EFFTIME_DARK_GFA'][mask]), np.sum(cat['EFFTIME_BRIGHT_GFA'][mask])))
    print()

                 ELG_EFFTIME_DARK  BGS_EFFTIME_BRIGHT   EFFTIME_DARK_GFA   EFFTIME_BRIGHT_GFA

LRGQSO 80605
subset 1: 6 exposures        3263                3351               3728                 3780

LRGQSO 80607
subset 1: 6 exposures        4046                3717               3834                 3705
subset 2: 6 exposures        4023                3666               3740                 3609

LRGQSO 80609
subset 1: 6 exposures        3841                3968               3754                 3863
subset 2: 7 exposures        3532                3650               3779                 3932

ELG 80606
subset 1: 7 exposures        4054                3977               4175                 4194

ELG 80608
subset 1: 5 exposures        4079                3664               3522                 3354
subset 2: 7 exposures        4589                4189               3954                 3916
subset 3: 5 exposures        4429                4072               3684                 3

In [8]:
# Print summary
for tileid_str in subsamp_dict.keys():
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    tile_flavor = cat['FAPRGRM'][mask][0].upper()
    n_exp = np.sum(mask)
    if tileid==80613:
        nom_depth = 180.
        depth_col = 'BGS_EFFTIME_BRIGHT'
    else:
        nom_depth = 1000.
        depth_col = 'EFFTIME_SPEC'
    tot_depth = np.sum(cat[depth_col][mask])
    print('Tile {} ({}):'.format(tileid, tile_flavor))
    print('all:       n_exp={:2}  {}={:5.0f}s'.format(n_exp, depth_col, tot_depth))
    subsets = subsamp_dict[tileid_str]
    for index, subset in enumerate(subsets):
        mask = np.in1d(cat['EXPID'], subset)
        subset_depth = np.sum(cat[depth_col][mask])
        print('subset {}:  n_exp={:2}  {}={:5.0f}s'.format(index+1, len(subset), depth_col, subset_depth))
    mask = (cat['TILEID']==tileid) & (~np.in1d(cat['EXPID'], np.concatenate(subsets)))
    unused_depth = np.sum(cat[depth_col][mask])
    print('unused:    n_exp={:2}  {}={:5.0f}s'.format(np.sum(mask), depth_col, unused_depth))
    print()

Tile 80605 (LRGQSO):
all:       n_exp=10  EFFTIME_SPEC= 5188s
subset 1:  n_exp= 7  EFFTIME_SPEC= 3263s
unused:    n_exp= 4  EFFTIME_SPEC= 1925s

Tile 80606 (ELG):
all:       n_exp=11  EFFTIME_SPEC= 6667s
subset 1:  n_exp= 7  EFFTIME_SPEC= 4054s
unused:    n_exp= 4  EFFTIME_SPEC= 2613s

Tile 80607 (LRGQSO):
all:       n_exp=15  EFFTIME_SPEC= 9450s
subset 1:  n_exp= 6  EFFTIME_SPEC= 4046s
subset 2:  n_exp= 6  EFFTIME_SPEC= 4023s
unused:    n_exp= 3  EFFTIME_SPEC= 1381s

Tile 80608 (ELG):
all:       n_exp=19  EFFTIME_SPEC=14986s
subset 1:  n_exp= 5  EFFTIME_SPEC= 4079s
subset 2:  n_exp= 7  EFFTIME_SPEC= 4589s
subset 3:  n_exp= 5  EFFTIME_SPEC= 4429s
unused:    n_exp= 2  EFFTIME_SPEC= 1889s

Tile 80609 (LRGQSO):
all:       n_exp=14  EFFTIME_SPEC= 7648s
subset 1:  n_exp= 6  EFFTIME_SPEC= 3841s
subset 2:  n_exp= 7  EFFTIME_SPEC= 3532s
unused:    n_exp= 1  EFFTIME_SPEC=  275s

Tile 80610 (ELG):
all:       n_exp=14  EFFTIME_SPEC= 9402s
subset 1:  n_exp= 7  EFFTIME_SPEC= 3950s
subset 2:  n_exp=